# AKConv 消融实验：YOLO11n-cls vs YOLO11n-cls-AKConv

**使用前提**：内核必须选择 `Python (yolov10)`（其余 kernel 未安装 torch）。

**运行方式**：从上到下依次运行每个代码块。想先快速验证流程就把 `EPOCHS` 改成 1。

## 实验设计

| 项目 | 配置 |
|---|---|
| 对照模型 | `yolo11n-cls.yaml`（原生 C3k2 模块） |
| 实验模型 | `yolo11n-cls-AKConv.yaml`（C3k2 换为 C3k2_AKConv） |
| 训练集 | `data/eyes_train`（5110 张，A/C/D/G/H/M/N/O 八类） |
| 验证集 | `data/eyes_val`（1163 张） |
| 评价指标 | top-1 / top-5 准确率、参数量、训练耗时 |

> 两个模型除骨干模块外结构与宽度完全一致，保证消融实验的公平性。
> 数据集直接位于项目 `data/` 目录下，通过 `data/eyes_dataset.yaml` 引用。


In [ ]:
import os, sys
print('Python:', sys.version.split()[0])

try:
    import torch
    print('torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('!! 未找到 torch —— 请把内核切换为 Python (yolov10) 后重新运行本单元')

In [ ]:
# 路径配置
PROJ = 'C:/Users/47449/Desktop/yolo/Intelligent_diagnosis_system'
ULTRA_DIR = os.path.join(PROJ, 'ultralytics-main')
DATA_ROOT = os.path.join(PROJ, 'data')
DATA_YAML = os.path.join(DATA_ROOT, 'eyes_dataset.yaml')
RUNS_DIR = os.path.join(PROJ, 'runs', 'ablation')

for name, p in [('项目根', PROJ), ('ultralytics-main', ULTRA_DIR),
                ('数据目录', DATA_ROOT), ('数据配置', DATA_YAML)]:
    print(('OK   ' if os.path.exists(p) else '缺失 ') + name + ': ' + p)

In [ ]:
# 让项目内的 ultralytics-main 生效（AKConv 注册在这里面）
if ULTRA_DIR not in sys.path:
    sys.path.insert(0, ULTRA_DIR)
os.chdir(PROJ)

import ultralytics
print('ultralytics 来源:', ultralytics.__file__)
assert 'Intelligent_diagnosis_system' in ultralytics.__file__, '导入的不是项目内的 ultralytics-main（AKConv 不可用）；请重启内核后重跑本单元'

from ultralytics import YOLO
from ultralytics.nn.modules.akconv import AKConv, C3k2_AKConv
print('AKConv / C3k2_AKConv 注册正常')

In [ ]:
# 数据集检查（数据已归位到项目 data/ 目录下）
for split in ['eyes_train', 'eyes_val']:
    p = os.path.join(DATA_ROOT, split)
    n = sum(len(f) for _, _, f in os.walk(p)) if os.path.exists(p) else 0
    print(split + ': ' + str(n) + ' 张')
    assert n > 0, split + ' 为空 —— 请确认数据集已放入 data/ 目录'

print('类别（字母序）:', sorted(os.listdir(os.path.join(DATA_ROOT, 'eyes_train'))))

In [ ]:
import time

EPOCHS = 50          # 先冒烟测试就改成 1
IMGSZ  = 224
BATCH  = 32
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

RESULTS = {}

def train_one(tag, model_yaml):
    print('=' * 60)
    print('训练 ' + tag + ': ' + model_yaml + '  epochs=' + str(EPOCHS))
    print('=' * 60)
    model = YOLO(model_yaml)
    n_param = sum(p.numel() for p in model.model.parameters())
    t0 = time.time()
    r = model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                    device=DEVICE, project=RUNS_DIR, name=tag, exist_ok=True)
    secs = time.time() - t0
    RESULTS[tag] = {
        'top1': getattr(r, 'top1', None),
        'top5': getattr(r, 'top5', None),
        'params': n_param,
        'seconds': secs,
    }
    print('>>> ' + tag + ' 完成：耗时 {:.0f}s'.format(secs))
    return RESULTS[tag]

print('配置：EPOCHS={}, IMGSZ={}, BATCH={}, DEVICE={}'.format(EPOCHS, IMGSZ, BATCH, DEVICE))

In [ ]:
# 对照模型：原生 YOLO11n-cls
train_one('baseline', 'yolo11n-cls.yaml')

In [ ]:
# 实验模型：YOLO11n-cls + AKConv
train_one('akconv', 'yolo11n-cls-AKConv.yaml')

In [ ]:
# 汇总对比
if len(RESULTS) == 2:
    rows = []
    for tag in ['baseline', 'akconv']:
        d = RESULTS[tag]
        rows.append({
            '模型': tag,
            'top-1 准确率(%)': round(d['top1'] * 100, 2) if d['top1'] is not None else None,
            'top-5 准确率(%)': round(d['top5'] * 100, 2) if d['top5'] is not None else None,
            '参数量': d['params'],
            '训练耗时(s)': round(d['seconds']),
        })
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for r in rows:
            print(r)
    if RESULTS['baseline']['top1'] is not None and RESULTS['akconv']['top1'] is not None:
        delta = (RESULTS['akconv']['top1'] - RESULTS['baseline']['top1']) * 100
        print()
        print('AKConv 相对 baseline 的 top-1 变化: {:+.2f} 个百分点'.format(delta))
else:
    print('本次未重跑，从历史结果读取：')

# 从历史训练结果读取（已训练完成，无需重跑）
import pandas as pd
for tag in ['baseline', 'akconv']:
    csv = os.path.join(RUNS_DIR, tag, 'results.csv')
    if os.path.exists(csv):
        df = pd.read_csv(csv)
        df.columns = [c.strip() for c in df.columns]
        t1 = [c for c in df.columns if 'top1' in c][0]
        print('[' + tag + '] epoch 数=' + str(len(df)) +
              '   最佳 top-1={:.2f}%'.format(df[t1].max() * 100))

## 结果解读与报告撰写提示

无论结果好坏都应如实记录，这本身就是合格的消融实验：

- **AKConv 提升**：说明可变形卷积的自适应采样对眼科图像中病灶形态变化更敏感。
- **AKConv 持平或下降**：说明在当前数据规模与分类任务上，偏移学习未带来净收益
  （可能原因：训练数据偏少、类别不均衡、偏移参数收敛不充分）。

### 可写入报告的数据

1. 上表的 top-1 / top-5 准确率对比
2. 参数量与理论计算量（GFLOPs）对比
3. 训练耗时对比（反映可变形卷积的额外计算开销）
4. `runs/ablation/*/results.csv` 中的逐 epoch 曲线，可绘制损失与准确率对比图

> 注意：`runs/` 目录已加入 `.gitignore`，训练产物不会进入仓库。
